# Feature Engineering for USD/LKR Exchange Rate Forecasting

## USD/LKR Exchange Rate Forecasting Under Economic Regime Changes

This notebook performs the feature-engineering stage of the USD/LKR exchange-rate forecasting project.

The objective is to transform the processed time-series data into a modelling-ready dataset by creating historical lag variables, lagged return variables, rolling statistical features, and a one-step-ahead forecasting target.

All predictor variables are constructed using information available up to the current time point in order to prevent data leakage.

## 1. Feature Engineering Overview

Feature engineering is an important stage in time-series forecasting because historical observations can be transformed into explanatory variables that allow forecasting models to learn temporal patterns.

The following groups of features will be created:

### Exchange-rate lag features
- Lag_1
- Lag_5
- Lag_20

### Return lag features
- Return_Lag_1
- Return_Lag_5
- Return_Lag_20

### Rolling statistical features
- Rolling_Mean_5
- Rolling_Mean_20
- Rolling_Std_20

### Forecasting target
- Target

The target represents the USD/LKR exchange rate at the next observation and therefore supports one-step-ahead forecasting.

## 2. Import Required Libraries

The required Python libraries are imported for data manipulation and numerical calculations.

In [1]:
import pandas as pd
import numpy as np

## 3. Load the EDA Dataset

The dataset generated during the exploratory data analysis stage is used as the input for feature engineering.

The dataset is loaded from the processed data directory.

It contains the cleaned USD/LKR exchange-rate series together with the derived variables created during the EDA stage.

In [2]:
df = pd.read_csv(
    "../data/processed/usd_lkr_eda.csv",
    parse_dates=["date"],
    index_col="date"
)

df.head()

,USD_LKR,log_return,rolling_volatility_20
date,,,
2010-05-07,113.6975,NaN,NaN
2010-05-11,113.6292,-0.000601,NaN
2010-05-12,113.6854,0.000494,NaN
2010-05-13,113.7009,0.000136,NaN
2010-05-14,113.6573,-0.000384,NaN


## 4. Initial Dataset Inspection

The structure and dimensions of the dataset are inspected before creating additional features.

In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (3499, 3)


In [4]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 3499 entries, 2010-05-07 to 2026-09-25
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   USD_LKR                3499 non-null   float64
 1   log_return             3498 non-null   float64
 2   rolling_volatility_20  3479 non-null   float64
dtypes: float64(3)
memory usage: 109.3 KB


In [5]:
df.head()

,USD_LKR,log_return,rolling_volatility_20
date,,,
2010-05-07,113.6975,NaN,NaN
2010-05-11,113.6292,-0.000601,NaN
2010-05-12,113.6854,0.000494,NaN
2010-05-13,113.7009,0.000136,NaN
2010-05-14,113.6573,-0.000384,NaN


In [6]:
df.tail()

,USD_LKR,log_return,rolling_volatility_20
date,,,
2026-09-21,330.3490,-0.005388,0.002268
2026-09-22,330.8125,0.001402,0.002171
2026-09-23,329.8867,-0.002802,0.002209
2026-09-24,329.0158,-0.002643,0.002271
2026-09-25,330.3522,0.004054,0.002423


## 5. Check Available Variables

The existing variables are inspected to confirm that the exchange-rate and return variables required for feature engineering are available.

In [7]:
print(df.columns.tolist())

['USD_LKR', 'log_return', 'rolling_volatility_20']


## 6. Sort Observations Chronologically

Since the dataset represents a time series, observations must be arranged in chronological order before lagged and rolling features are created.

Maintaining chronological order is essential because lag features must represent historical observations rather than future observations.

In [8]:
df = df.sort_index()

In [9]:
df.head()

,USD_LKR,log_return,rolling_volatility_20
date,,,
2010-05-07,113.6975,NaN,NaN
2010-05-11,113.6292,-0.000601,NaN
2010-05-12,113.6854,0.000494,NaN
2010-05-13,113.7009,0.000136,NaN
2010-05-14,113.6573,-0.000384,NaN


## 7. Check for Duplicate Dates

Duplicate dates are checked before creating time-dependent features.

Duplicate observations can cause problems when calculating lagged and rolling variables.

In [10]:
print("Number of duplicate dates:", df.index.duplicated().sum())

Number of duplicate dates: 0


## 8. Check Missing Values Before Feature Engineering

Existing missing values are examined before creating new variables.

This helps determine whether missing observations already exist in the processed dataset.

In [11]:
df.isna().sum()

USD_LKR                   0
log_return                1
rolling_volatility_20    20
dtype: int64

## 9. Exchange-Rate Lag Features

Lagged exchange-rate variables are created to capture the dependence of the current exchange rate on its previous observations.

Three lag periods are used:

- **Lag_1** represents the exchange rate at the previous observation.
- **Lag_5** represents the exchange rate five observations earlier.
- **Lag_20** represents the exchange rate twenty observations earlier.

Only historical observations are used in these variables. Therefore, the lag features do not use future information.

In [12]:
df["Lag_1"] = df["USD_LKR"].shift(1)

df["Lag_5"] = df["USD_LKR"].shift(5)

df["Lag_20"] = df["USD_LKR"].shift(20)

## 10. Inspect Exchange-Rate Lag Features

The newly created lag variables are inspected to verify that they correctly represent previous exchange-rate observations.

In [13]:
df[
    [
        "USD_LKR",
        "Lag_1",
        "Lag_5",
        "Lag_20"
    ]
].head(25)

,USD_LKR,Lag_1,Lag_5,Lag_20
date,,,,
2010-05-07,113.6975,NaN,NaN,NaN
2010-05-11,113.6292,113.6975,NaN,NaN
2010-05-12,113.6854,113.6292,NaN,NaN
2010-05-13,113.7009,113.6854,NaN,NaN
2010-05-14,113.6573,113.7009,NaN,NaN
2010-06-01,113.8271,113.6573,113.6975,NaN
2010-06-02,113.7567,113.8271,113.6292,NaN
2010-06-14,113.5446,113.7567,113.6854,NaN
2010-06-15,113.5720,113.5446,113.7009,NaN


## 11. Lagged Log-Return Features

Lagged daily logarithmic returns are created to capture short-term dependence in exchange-rate changes.

The following variables are created:

- **Return_Lag_1**
- **Return_Lag_5**
- **Return_Lag_20**

These variables contain only historical return information.

In [14]:
df["Return_Lag_1"] = df["log_return"].shift(1)

df["Return_Lag_5"] = df["log_return"].shift(5)

df["Return_Lag_20"] = df["log_return"].shift(20)

## 12. Inspect Lagged Return Features

The lagged return variables are inspected to confirm that they contain historical return information.

In [15]:
df[
    [
        "log_return",
        "Return_Lag_1",
        "Return_Lag_5",
        "Return_Lag_20"
    ]
].head(25)

,log_return,Return_Lag_1,Return_Lag_5,Return_Lag_20
date,,,,
2010-05-07,NaN,NaN,NaN,NaN
2010-05-11,-0.000601,NaN,NaN,NaN
2010-05-12,0.000494,-0.000601,NaN,NaN
2010-05-13,0.000136,0.000494,NaN,NaN
2010-05-14,-0.000384,0.000136,NaN,NaN
2010-06-01,0.001493,-0.000384,NaN,NaN
2010-06-02,-0.000619,0.001493,-0.000601,NaN
2010-06-14,-0.001866,-0.000619,0.000494,NaN
2010-06-15,0.000241,-0.001866,0.000136,NaN


## 13. Rolling Mean Features

Rolling mean features are created to capture the recent level and local trend of the USD/LKR exchange rate.

Two rolling windows are used:

- **Rolling_Mean_5**: mean exchange rate over the previous five observations including the current observation.
- **Rolling_Mean_20**: mean exchange rate over the previous twenty observations including the current observation.

The rolling calculations are based only on information available at the current time point.

In [16]:
df["Rolling_Mean_5"] = (
    df["USD_LKR"]
    .rolling(window=5)
    .mean()
)

df["Rolling_Mean_20"] = (
    df["USD_LKR"]
    .rolling(window=20)
    .mean()
)

## 14. Inspect Rolling Mean Features

The rolling mean variables are inspected to verify that the calculations have been applied correctly.

In [17]:
df[
    [
        "USD_LKR",
        "Rolling_Mean_5",
        "Rolling_Mean_20"
    ]
].head(25)

,USD_LKR,Rolling_Mean_5,Rolling_Mean_20
date,,,
2010-05-07,113.6975,NaN,NaN
2010-05-11,113.6292,NaN,NaN
2010-05-12,113.6854,NaN,NaN
2010-05-13,113.7009,NaN,NaN
2010-05-14,113.6573,113.67406,NaN
2010-06-01,113.8271,113.69998,NaN
2010-06-02,113.7567,113.72548,NaN
2010-06-14,113.5446,113.69732,NaN
2010-06-15,113.5720,113.67154,NaN


## 15. Rolling Standard Deviation

A 20-observation rolling standard deviation is created to represent the recent variability of the USD/LKR exchange-rate level.

Higher values indicate greater variation in the exchange-rate observations within the rolling window, while lower values indicate relatively lower variation.

In [18]:
df["Rolling_Std_20"] = (
    df["USD_LKR"]
    .rolling(window=20)
    .std()
)

## 16. Inspect Rolling Standard Deviation

The 20-observation rolling standard deviation is inspected after feature creation.

In [19]:
df[
    [
        "USD_LKR",
        "Rolling_Std_20"
    ]
].head(25)

,USD_LKR,Rolling_Std_20
date,,
2010-05-07,113.6975,NaN
2010-05-11,113.6292,NaN
2010-05-12,113.6854,NaN
2010-05-13,113.7009,NaN
2010-05-14,113.6573,NaN
2010-06-01,113.8271,NaN
2010-06-02,113.7567,NaN
2010-06-14,113.5446,NaN
2010-06-15,113.5720,NaN


## 17. Review the Engineered Features

All newly created features are inspected together to confirm that the required variables have been generated successfully.

In [20]:
feature_columns = [
    "Lag_1",
    "Lag_5",
    "Lag_20",
    "Return_Lag_1",
    "Return_Lag_5",
    "Return_Lag_20",
    "Rolling_Mean_5",
    "Rolling_Mean_20",
    "Rolling_Std_20"
]

df[feature_columns].head(25)

,Lag_1,Lag_5,Lag_20,Return_Lag_1,Return_Lag_5,Return_Lag_20,Rolling_Mean_5,Rolling_Mean_20,Rolling_Std_20
date,,,,,,,,,
2010-05-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-05-11,113.6975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-05-12,113.6292,NaN,NaN,-0.000601,NaN,NaN,NaN,NaN,NaN
2010-05-13,113.6854,NaN,NaN,0.000494,NaN,NaN,NaN,NaN,NaN
2010-05-14,113.7009,NaN,NaN,0.000136,NaN,NaN,113.67406,NaN,NaN
2010-06-01,113.6573,113.6975,NaN,-0.000384,NaN,NaN,113.69998,NaN,NaN
2010-06-02,113.8271,113.6292,NaN,0.001493,-0.000601,NaN,113.72548,NaN,NaN
2010-06-14,113.7567,113.6854,NaN,-0.000619,0.000494,NaN,113.69732,NaN,NaN
2010-06-15,113.5446,113.7009,NaN,-0.001866,0.000136,NaN,113.67154,NaN,NaN


## 18. Missing Values Created by Lag and Rolling Features

Lagged and rolling calculations naturally create missing values at the beginning of the dataset.

For example, a 20-observation lag cannot be calculated for the first 20 observations because sufficient historical information is unavailable.

These missing observations will be handled after all required features and the forecasting target have been created.

In [21]:
df.isna().sum()

USD_LKR                   0
log_return                1
rolling_volatility_20    20
Lag_1                     1
Lag_5                     5
Lag_20                   20
Return_Lag_1              2
Return_Lag_5              6
Return_Lag_20            21
Rolling_Mean_5            4
Rolling_Mean_20          19
Rolling_Std_20           19
dtype: int64

## 19. Create the One-Step-Ahead Forecasting Target

The forecasting problem is formulated as a one-step-ahead prediction problem.

For each observation at time **t**, the objective is to predict the USD/LKR exchange rate at the next observation **t+1**.

Therefore:

**Target(t) = USD_LKR(t+1)**

The target is created by shifting the USD/LKR exchange-rate series one observation backward.

The predictor variables continue to contain only information available at time t, while the target represents the future value to be predicted.

In [22]:
df["Target"] = df["USD_LKR"].shift(-1)

## 20. Inspect the Forecasting Target

The current USD/LKR exchange rate and the next-observation target are compared to verify that the target represents the following exchange-rate observation.

In [23]:
df[
    [
        "USD_LKR",
        "Target"
    ]
].head(10)

,USD_LKR,Target
date,,
2010-05-07,113.6975,113.6292
2010-05-11,113.6292,113.6854
2010-05-12,113.6854,113.7009
2010-05-13,113.7009,113.6573
2010-05-14,113.6573,113.8271
2010-06-01,113.8271,113.7567
2010-06-02,113.7567,113.5446
2010-06-14,113.5446,113.5720
2010-06-15,113.5720,113.4625


## 21. Check the Final Target Observation

The final observation does not have a future exchange-rate value available because there is no observation after the final date.

Therefore, the final target value is expected to be missing.

In [24]:
df.tail()

,USD_LKR,log_return,rolling_volatility_20,Lag_1,Lag_5,Lag_20,Return_Lag_1,Return_Lag_5,Return_Lag_20,Rolling_Mean_5,Rolling_Mean_20,Rolling_Std_20,Target
date,,,,,,,,,,,,,
2026-09-21,330.3490,-0.005388,0.002268,332.1338,328.6751,331.2303,0.000728,0.000239,-0.001814,330.71720,329.153645,1.207957,330.8125
2026-09-22,330.8125,0.001402,0.002171,330.3490,329.0351,330.1657,-0.005388,0.001095,-0.003219,331.07268,329.185985,1.244582,329.8867
2026-09-23,329.8867,-0.002802,0.002209,330.8125,330.1761,329.4456,0.001402,0.003462,-0.002183,331.01480,329.208040,1.253302,329.0158
2026-09-24,329.0158,-0.002643,0.002271,329.8867,331.8920,329.0037,-0.002802,0.005183,-0.001342,330.43956,329.208645,1.253201,330.3522
2026-09-25,330.3522,0.004054,0.002423,329.0158,332.1338,328.6088,-0.002643,0.000728,-0.001201,330.08324,329.295815,1.269805,NaN


## 22. Review Missing Values Before Cleaning

All missing values generated during feature engineering are reviewed before the final modelling dataset is created.

In [25]:
df.isna().sum()

USD_LKR                   0
log_return                1
rolling_volatility_20    20
Lag_1                     1
Lag_5                     5
Lag_20                   20
Return_Lag_1              2
Return_Lag_5              6
Return_Lag_20            21
Rolling_Mean_5            4
Rolling_Mean_20          19
Rolling_Std_20           19
Target                    1
dtype: int64

## 23. Create the Final Modelling Dataset

After all lagged variables, rolling variables, and the one-step-ahead target have been created, observations with missing values are removed.

This removes:

1. Initial observations that do not have sufficient historical information for the lagged and rolling features.
2. The final observation, for which a one-step-ahead target does not exist.

The resulting dataset contains complete observations suitable for the subsequent modelling stage.

In [26]:
df_model = df.dropna().copy()

## 24. Verify the Final Dataset

The final modelling dataset is checked to confirm that no missing values remain.

In [27]:
df_model.isna().sum()

USD_LKR                  0
log_return               0
rolling_volatility_20    0
Lag_1                    0
Lag_5                    0
Lag_20                   0
Return_Lag_1             0
Return_Lag_5             0
Return_Lag_20            0
Rolling_Mean_5           0
Rolling_Mean_20          0
Rolling_Std_20           0
Target                   0
dtype: int64

## 25. Final Dataset Dimensions

The dimensions of the feature-engineered dataset are examined to determine the number of observations and variables available for modelling.

In [28]:
print("Final dataset shape:", df_model.shape)

Final dataset shape: (3477, 13)


In [29]:
df_model.head()

,USD_LKR,log_return,rolling_volatility_20,Lag_1,Lag_5,Lag_20,Return_Lag_1,Return_Lag_5,Return_Lag_20,Rolling_Mean_5,Rolling_Mean_20,Rolling_Std_20,Target
date,,,,,,,,,,,,,
2010-10-13,111.9335,-0.003326,0.001652,112.3064,112.6897,113.6292,-0.002115,0.000745,-0.000601,112.38808,113.041855,0.602375,111.5154
2010-11-09,111.5154,-0.003742,0.001753,111.9335,112.6132,113.6854,-0.003326,-0.000679,0.000494,112.16852,112.933355,0.671790,111.5843
2010-11-10,111.5843,0.000618,0.001772,111.5154,112.5431,113.7009,-0.003742,-0.000623,0.000136,111.97676,112.827525,0.710135,111.6845
2010-11-11,111.6845,0.000898,0.001816,111.5843,112.5442,113.6573,0.000618,0.000010,-0.000384,111.80482,112.728885,0.725655,111.6000
2010-11-12,111.6000,-0.000757,0.001729,111.6845,112.3064,113.8271,0.000898,-0.002115,0.001493,111.66354,112.617530,0.719109,111.6850


## 26. Final Feature Set

The final feature-engineered dataset contains the original exchange-rate information, historical lag variables, return lag variables, rolling statistical features, and the one-step-ahead forecasting target.

In [30]:
print(df_model.columns.tolist())

['USD_LKR', 'log_return', 'rolling_volatility_20', 'Lag_1', 'Lag_5', 'Lag_20', 'Return_Lag_1', 'Return_Lag_5', 'Return_Lag_20', 'Rolling_Mean_5', 'Rolling_Mean_20', 'Rolling_Std_20', 'Target']


## 27. Define Predictor Variables

The predictor variables used for machine-learning forecasting are explicitly defined.

These variables represent historical exchange-rate behaviour, historical returns, and recent rolling statistics.

The current target variable is kept separate from the predictor variables.

In [31]:
features = [
    "Lag_1",
    "Lag_5",
    "Lag_20",
    "Return_Lag_1",
    "Return_Lag_5",
    "Return_Lag_20",
    "Rolling_Mean_5",
    "Rolling_Mean_20",
    "Rolling_Std_20"
]

## 28. Separate Predictors and Target

The predictor matrix **X** contains the engineered historical features.

The target vector **y** contains the next-observation USD/LKR exchange rate.

This separation prepares the dataset for the supervised machine-learning models used in the subsequent forecasting stage.

In [32]:
X = df_model[features]

y = df_model["Target"]

In [33]:
X.head()

,Lag_1,Lag_5,Lag_20,Return_Lag_1,Return_Lag_5,Return_Lag_20,Rolling_Mean_5,Rolling_Mean_20,Rolling_Std_20
date,,,,,,,,,
2010-10-13,112.3064,112.6897,113.6292,-0.002115,0.000745,-0.000601,112.38808,113.041855,0.602375
2010-11-09,111.9335,112.6132,113.6854,-0.003326,-0.000679,0.000494,112.16852,112.933355,0.671790
2010-11-10,111.5154,112.5431,113.7009,-0.003742,-0.000623,0.000136,111.97676,112.827525,0.710135
2010-11-11,111.5843,112.5442,113.6573,0.000618,0.000010,-0.000384,111.80482,112.728885,0.725655
2010-11-12,111.6845,112.3064,113.8271,0.000898,-0.002115,0.001493,111.66354,112.617530,0.719109


In [34]:
y.head()

date
2010-10-13    111.5154
2010-11-09    111.5843
2010-11-10    111.6845
2010-11-11    111.6000
2010-11-12    111.6850
Name: Target, dtype: float64

## 29. Check Predictor and Target Dimensions

The dimensions of X and y are checked to ensure that both contain the same number of observations.

In [35]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3477, 9)
y shape: (3477,)


## 30. Verify Chronological Order

The final modelling dataset is checked to ensure that observations remain in chronological order.

Maintaining chronological order is essential because the dataset represents a time series and future observations must not be used to predict earlier observations.

In [36]:
print("First date:", df_model.index.min())
print("Last date:", df_model.index.max())

First date: 2010-10-13 00:00:00
Last date: 2026-09-24 00:00:00


In [37]:
print("Data is sorted:", df_model.index.is_monotonic_increasing)

Data is sorted: True


## 31. Data Leakage Check

Data leakage occurs when information from the future is unintentionally used to construct predictor variables.

The feature-engineering process avoids future information by:

- creating lag variables using previous observations,
- calculating rolling statistics using information available at the current observation,
- creating the forecasting target separately using the next observation,
- maintaining chronological ordering.

Therefore, the predictor variables represent information that would have been available at the time the forecast was made.

In [38]:
leakage_check = df_model[features].isna().sum()

print("Missing values in predictor variables:")
print(leakage_check)

Missing values in predictor variables:
Lag_1              0
Lag_5              0
Lag_20             0
Return_Lag_1       0
Return_Lag_5       0
Return_Lag_20      0
Rolling_Mean_5     0
Rolling_Mean_20    0
Rolling_Std_20     0
dtype: int64


## 32. Save the Feature-Engineered Dataset

The completed feature-engineered dataset is saved in the processed data directory.

This dataset will be used as the input for the forecasting-model development stage.

In [39]:
df_model.to_csv(
    "../data/processed/feature_engineered_dataset.csv"
)

## 33. Verify the Saved Dataset

The saved feature-engineered dataset is reloaded to verify that it was successfully written to the processed data directory.

In [40]:
check_df = pd.read_csv(
    "../data/processed/feature_engineered_dataset.csv",
    parse_dates=["date"],
    index_col="date"
)

print("Saved dataset shape:", check_df.shape)

Saved dataset shape: (3477, 13)


In [41]:
check_df.head()

,USD_LKR,log_return,rolling_volatility_20,Lag_1,Lag_5,Lag_20,Return_Lag_1,Return_Lag_5,Return_Lag_20,Rolling_Mean_5,Rolling_Mean_20,Rolling_Std_20,Target
date,,,,,,,,,,,,,
2010-10-13,111.9335,-0.003326,0.001652,112.3064,112.6897,113.6292,-0.002115,0.000745,-0.000601,112.38808,113.041855,0.602375,111.5154
2010-11-09,111.5154,-0.003742,0.001753,111.9335,112.6132,113.6854,-0.003326,-0.000679,0.000494,112.16852,112.933355,0.671790,111.5843
2010-11-10,111.5843,0.000618,0.001772,111.5154,112.5431,113.7009,-0.003742,-0.000623,0.000136,111.97676,112.827525,0.710135,111.6845
2010-11-11,111.6845,0.000898,0.001816,111.5843,112.5442,113.6573,0.000618,0.000010,-0.000384,111.80482,112.728885,0.725655,111.6000
2010-11-12,111.6000,-0.000757,0.001729,111.6845,112.3064,113.8271,0.000898,-0.002115,0.001493,111.66354,112.617530,0.719109,111.6850


# 34. Feature Engineering Summary

The feature-engineering stage transformed the processed USD/LKR exchange-rate dataset into a modelling-ready dataset.

The following historical exchange-rate features were created:

- Lag_1
- Lag_5
- Lag_20

The following lagged return features were created:

- Return_Lag_1
- Return_Lag_5
- Return_Lag_20

The following rolling statistical features were created:

- Rolling_Mean_5
- Rolling_Mean_20
- Rolling_Std_20

A one-step-ahead forecasting target was created by shifting the USD/LKR exchange-rate series by one observation.

Rows with insufficient historical information and the final observation without a future target were removed.

The final dataset was checked for missing values, chronological ordering, and potential data leakage.

The completed feature-engineered dataset was saved as:

`feature_engineered_dataset.csv`

This dataset will be used in the subsequent forecasting-model development stage.